<a href="https://colab.research.google.com/github/Josue-Martinez10/Dev2/blob/Agentic-AI/118s_Capstone_Project_Solo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
%%capture
%pip install --upgrade --quiet langchain langchain-core langchain-openai langchain-community langgraph pypdf langchain-chroma openpyxl chromadb

In [38]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
from openai import OpenAI
from google.colab import userdata


api_key = userdata.get("OpenAi")
os.environ["OPENAI_API_KEY"] = api_key


model = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)


embedding = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

In [40]:
import time
import requests
from bs4 import BeautifulSoup
import faiss
import numpy as np
from openai import OpenAI

client = OpenAI()


# AI Classifier Agent Using AI Prompt
def classify_issue_ai(description):
    prompt = f"""
    You are an IT helpdesk classifier.
    Categorize the user's IT issue into ONE category from this list:

    - Account / Login Issue
    - Network Issue
    - Hardware / Performance Issue
    - Email Issue
    - Software Issue
    - Security Issue
    - Printer Issue
    - General IT Issue

    User description: "{description}"

    Respond ONLY with the category.
    """
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=20
    )
    return response.choices[0].message.content.strip()

# Knowledge Agent Using Website RAG

class KnowledgeAgent:
    def __init__(self, url):
        self.url = url
        self.web_text = self.fetch_website_text(url)
        self.chunks = self.chunk_text(self.web_text, chunk_size=500)
        self.index = self.build_vector_index(self.chunks)

    def fetch_website_text(self, url):
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style"]):
            tag.extract()
        text = soup.get_text(separator=" ")
        return " ".join(text.split())

    def chunk_text(self, text, chunk_size=500):
        words = text.split()
        return [
            " ".join(words[i:i + chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    def embed(self, texts):
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )
        return [np.array(x.embedding, dtype="float32") for x in response.data]

    def build_vector_index(self, chunks):
        embeddings = self.embed(chunks)
        dim = len(embeddings[0])
        index = faiss.IndexFlatL2(dim)
        index.add(np.stack(embeddings))
        self.embeddings = embeddings
        return index

    def search(self, query, k=3):
        query_vec = self.embed([query])[0].reshape(1, -1)
        scores, indices = self.index.search(query_vec, k)
        return [self.chunks[i] for i in indices[0]]

    def answer(self, query):
        retrieved = self.search(query)
        prompt = f"""
        You are an IT Knowledge Agent.
        Use ONLY the following retrieved text from the website to answer the user's question:

        Retrieved info:
        {retrieved}

        User question: "{query}"

        Provide a short, accurate response using ONLY the retrieved text.
        """
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=250
        )
        return response.choices[0].message.content

# Workflow Agent using Keywords
class WorkflowAgent:
    def reset_user_account(self, username):
        print(f"[Workflow] Resetting account for user: {username}...")
        time.sleep(1)
        return f"Account for {username} has been reset."

    def unlock_user(self, username):
        print(f"[Workflow] Unlocking user: {username}...")
        time.sleep(1)
        return f"User {username} has been unlocked."

    def check_system_logs(self, service):
        print(f"[Workflow] Checking logs for: {service}...")
        time.sleep(1)
        return f"No critical errors found in logs for {service}."

    def restart_service(self, service):
        print(f"[Workflow] Restarting service: {service}...")
        time.sleep(1)
        return f"Service '{service}' restarted successfully."

    def execute(self, action, target):
        if action == "reset_account":
            return self.reset_user_account(target)
        elif action == "unlock_user":
            return self.unlock_user(target)
        elif action == "check_logs":
            return self.check_system_logs(target)
        elif action == "restart_service":
            return self.restart_service(target)
        else:
            return f"Unknown workflow action: '{action}'"

# Main Intake + Multi-Agent Pipeline

def intake_agent():
    print("=== IT Support Intake ===")
    name = input("Your name: ")
    department = input("Your department: ")
    issue_description = input("Describe your IT problem: ")

    # 1. AI Classification
    category = classify_issue_ai(issue_description)

    # 2. Knowledge Retrieval with Website
    knowledge_url = "https://www.ratcliff.it/news/top-15-most-common-it-issues"
    kn_agent = KnowledgeAgent(knowledge_url)
    helpful_answer = kn_agent.answer(issue_description)

    # 3. Workflow Automation using keywords
    wf_agent = WorkflowAgent()
    automation_output = None

    issue_lower = issue_description.lower()
    if "password" in issue_lower:
        automation_output = wf_agent.execute("reset_account", name)
    elif "locked" in issue_lower:
        automation_output = wf_agent.execute("unlock_user", name)
    elif "vpn" in issue_lower:
        automation_output = wf_agent.execute("check_logs", "vpn_service")
    elif "email" in issue_lower:
        automation_output = wf_agent.execute("restart_service", "email_service")
    elif "network" in issue_lower:
        automation_output = wf_agent.execute("check_logs", "network_service")
    elif "hardware" in issue_lower:
        automation_output = wf_agent.execute("restart_service", "hardware_service")
    elif "turn on" in issue_lower:
        automation_output = wf_agent.execute("restart_service", "turn_on_service")

# Summary Output
    print("\n=== Summary ===")
    print(f"Name: {name}")
    print(f"Department: {department}")
    print(f"Issue Description: {issue_description}")
    print(f"AI Category: {category}")

    print("\n=== Knowledge Agent Response ===")
    print(helpful_answer)

    print("\n=== Workflow Agent Execution ===")
    if automation_output:
        print(automation_output)
    else:
        print("No automation triggered for this issue.")


if __name__ == "__main__":
    intake_agent()


=== IT Support Intake ===
Your name: Josue
Your department: Marketing
Describe your IT problem: My computer is locked
[Workflow] Unlocking user: Josue...

=== Summary ===
Name: Josue
Department: Marketing
Issue Description: My computer is locked
AI Category: Account / Login Issue

=== Knowledge Agent Response ===
If your computer is locked due to password or access issues, it might be caused by weak or forgotten passwords. To fix this, use a password manager to securely store passwords and follow strong password rules, including multi-step verification for added security. If the lock is due to system crashes or software failures, try restarting your system to clear memory and improve performance.

=== Workflow Agent Execution ===
User Josue has been unlocked.
